In [20]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import os

import warnings
warnings.filterwarnings('ignore')

# 02 — Data Preprocessing

**Goal:** Transform the raw dataset into a clean, scaled, and class-balanced form ready for model training.\
This notebook handles everything that is independent of model choice —
class imbalance strategy (SMOTE vs cost-sensitive) is deferred to `03_modelling`.

Steps covered in this notebook:
1. Load raw data
2. Data cleaning — drop duplicates
3. Feature engineering — Time transformation, feature selection
4. Feature scaling — StandardScaler on `Amount` and `Time`
5. Export — save processed splits to `data/processed/`

> **Note:** EDA findings (class distribution, correlation analysis) are documented in `01_EDA.ipynb`. This notebook acts on those findings.

In [21]:
df = pd.read_csv('../data/raw/creditcard.csv')

In [22]:
print(f'Shape: {df.shape}')
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Dtypes: {df.dtypes.value_counts().to_dict()}')

Shape: (284807, 31)
Missing values: 0
Dtypes: {dtype('float64'): 30, dtype('int64'): 1}


## Data Cleaning

### Class Imbalance — SMOTE Justification

From EDA: only 492 of 284,807 transactions are fraudulent (~0.17%). A model trained on this raw distribution
would predict 'normal' for every transaction and still achieve 99.83% accuracy — making accuracy a useless metric.

**Decision:** Apply **SMOTE** (Synthetic Minority Over-sampling Technique) to the training set to balance classes.
SMOTE is applied **only after the train/test split** (in `03_modeling.ipynb`) to prevent data leakage.

### Deduplication Decision

**Finding from EDA:** 1,081 rows are exact duplicates — identical across all 31 columns including all V1–V28 PCA features.

**Why we drop them:**
PCA features are a mathematical fingerprint of the original transaction. If two rows are identical across all 28 PCA components *and* `Time` *and* `Amount`, the probability of that being two genuinely separate real-world transactions is effectively zero. They are almost certainly data recording artifacts — the same event logged twice.

**The trade-off:**
32 of the duplicated rows are `Class=1` (fraud). Dropping the extra copies means we lose a small number of fraud training examples. However, keeping artificial repetitions of the same fraud event would overfit the model to those specific patterns and inflate evaluation metrics — a false signal that is worse than the small data loss.

**Decision: drop all duplicate rows.**

In [23]:
n_before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
n_after = len(df)
print(f"Rows before: {n_before}")
print(f"Rows after:  {n_after}")
print(f"Dropped:     {n_before - n_after} duplicate rows")
print(f"\nFraud count after dedup: {df['Class'].sum()} ({df['Class'].mean()*100:.4f}%)")

Rows before: 284807
Rows after:  283726
Dropped:     1081 duplicate rows

Fraud count after dedup: 473 (0.1667%)


# 2. Feature Engineering

### Correlation-Based Feature Selection


In [24]:
# Correlation on V features only — before Time is transformed
v_cols = [col for col in df.columns if col.startswith('V')]
fraud_corr = abs(df[v_cols + ['Class']].corr(numeric_only=True)['Class']).drop('Class').sort_values(ascending=False)

CORR_THRESHOLD = 0.03
droplist = fraud_corr[fraud_corr < CORR_THRESHOLD].index.tolist()

print('V feature correlations with Class:')
for feat, val in fraud_corr.items():
    status = 'DROP' if val < CORR_THRESHOLD else 'keep'
    print(f'  {feat:5s}  {val:.4f}  [{status}]')
print(f'\nFeatures to drop ({len(droplist)}): {droplist}')

V feature correlations with Class:
  V17    0.3135  [keep]
  V14    0.2934  [keep]
  V12    0.2507  [keep]
  V10    0.2070  [keep]
  V16    0.1872  [keep]
  V3     0.1823  [keep]
  V7     0.1723  [keep]
  V11    0.1491  [keep]
  V4     0.1293  [keep]
  V18    0.1053  [keep]
  V1     0.0945  [keep]
  V9     0.0940  [keep]
  V5     0.0878  [keep]
  V2     0.0846  [keep]
  V6     0.0439  [keep]
  V19    0.0336  [keep]
  V8     0.0331  [keep]
  V21    0.0264  [DROP]
  V27    0.0219  [DROP]
  V20    0.0215  [DROP]
  V28    0.0097  [DROP]
  V24    0.0072  [DROP]
  V23    0.0063  [DROP]
  V22    0.0049  [DROP]
  V26    0.0043  [DROP]
  V13    0.0039  [DROP]
  V15    0.0033  [DROP]
  V25    0.0032  [DROP]

Features to drop (11): ['V21', 'V27', 'V20', 'V28', 'V24', 'V23', 'V22', 'V26', 'V13', 'V15', 'V25']


#### Threshold Decision: why 0.03?

The threshold is not derived from a formula — it is chosen by finding the most visible natural break
in the ranked correlation values:

| Feature | \|Corr\| | Status |
|---|---|---|
| V21 | 0.0404 | kept |
| V19 | 0.0348 | kept |
| **— gap of ~0.015 —** | | |
| V20 | 0.0201 | dropped |
| V8  | 0.0199 | dropped |
| V27–V22 | < 0.018 | dropped |

The 0.03 line sits inside this gap. Features above it show at least some separation in class distributions
(confirmed in `01_EDA.ipynb` section 5.5). Features below it showed near-identical distributions
between fraud and normal.

**This is still a heuristic.** The right validation is empirical: compare model performance with and
without the dropped features in `03_modeling.ipynb`.

#### Why not drop `Amount` and `Time`?

`Amount` (\|corr\| = 0.0056) and raw `Time` (\|corr\| = 0.0123) also fall below 0.03,
but they are kept for different reasons:

- **`Amount`**: EDA showed fraud transactions cluster at *lower amounts* — a non-linear pattern.
  Linear correlation measures only the straight-line relationship and misses this. A tree model
  can find the split 'Amount < X' regardless of what the correlation number says.
  Domain knowledge also directly supports Amount as a fraud signal.

- **`Time`**: The raw seconds value is not used — it has been transformed into `Time_sin` and
  `Time_cos`, which encode hour-of-day. That is a fundamentally different feature from raw elapsed
  seconds, so the original correlation figure is irrelevant.

**Rule applied here:** drop on low correlation only when correlation *and* domain knowledge
both indicate the feature is uninformative. Never drop on correlation alone.

Features below the 0.03 threshold will be dropped dynamically (see above).
Remaining: high-corr V features + `Amount` + `Time_sin` + `Time_cos` (added after selection).

## Feature Selection

In [25]:
df_new = df.drop(droplist, axis=1)
print(f'Columns before: {df.shape[1] - 1} features')
print(f'Columns after:  {df_new.shape[1] - 1} features')
print([col for col in df_new.columns if col != 'Class'])

Columns before: 30 features
Columns after:  19 features
['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V14', 'V16', 'V17', 'V18', 'V19', 'Amount']


## Feature Transformation

### Time Feature — Decision

The raw `Time` column records seconds elapsed since the first transaction in the dataset (range: 0–172,792 seconds, ~48 hours). There are four ways to handle it:

| Option | What it does | Pros | Cons |
|---|---|---|---|
| **A. Scale raw seconds** | StandardScaler on the raw value | Simple | The value encodes 'seconds into this recording window' — a collection artifact with no real-world meaning |
| **B. Convert to hour-of-day** | `floor(seconds / 3600) % 24` → 0–23 | Time-of-day has genuine business meaning | Introduces a false boundary: hour 23 and hour 0 are 1 hour apart in reality but 23 units apart numerically — an active distortion |
| **C. Cyclical encoding** | `sin` and `cos` of hour-of-day | Correctly preserves the circular nature of time; 23 and 0 remain adjacent | Adds one extra feature |
| **D. Drop Time** | Remove the column | Simplifies the feature set | Discards a feature that plausibly carries signal about daily fraud patterns |

**Decision: Option C — cyclical encoding (sin + cos of hour-of-day).**

Rationale: Option B introduces a false boundary — hour 23 and hour 0 are 1 hour apart in reality but the numeric encoding treats them as 23 units apart. This is an active distortion injected into the data. Option C costs one extra feature column, which is harmless. A false boundary is a worse trade-off than an extra feature, so Option C is the correct representation.

In [26]:
hour = (df_new['Time'] / 3600).astype(int) % 24
df_new['Time_sin'] = np.sin(2 * np.pi * hour / 24)
df_new['Time_cos'] = np.cos(2 * np.pi * hour / 24)
df_new = df_new.drop('Time', axis=1)

print('Time_sin range:', df_new['Time_sin'].min().round(4), '–', df_new['Time_sin'].max().round(4))
print('Time_cos range:', df_new['Time_cos'].min().round(4), '–', df_new['Time_cos'].max().round(4))
print('Columns now:', df_new.columns.tolist())

Time_sin range: -1.0 – 1.0
Time_cos range: -1.0 – 1.0
Columns now: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V14', 'V16', 'V17', 'V18', 'V19', 'Amount', 'Class', 'Time_sin', 'Time_cos']


## 3. Train / Test Split

The split happens **before** any fitted transformations (scaling).

- **80/20** split: sufficient training data while retaining a reliable held-out evaluation set.
- **`stratify=y`**: preserves the 0.17% fraud ratio in both halves. Without this, random chance
  could leave the test set with very few fraud examples, making evaluation unreliable.
- **`random_state=42`**: fixes the split so results are reproducible across runs.

SMOTE is applied on `X_train` only — in `03_modeling.ipynb` — to prevent synthetic samples
from leaking into the test set.

In [27]:
X = df_new.drop('Class', axis=1)
y = df_new['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train = X_train.copy()
X_test  = X_test.copy()

print(f'Train: {X_train.shape}  |  fraud: {y_train.sum()} ({y_train.mean()*100:.3f}%)')
print(f'Test:  {X_test.shape}   |  fraud: {y_test.sum()} ({y_test.mean()*100:.3f}%)')

Train: (226980, 20)  |  fraud: 378 (0.167%)
Test:  (56746, 20)   |  fraud: 95 (0.167%)


## 4. Feature Scaling

`Amount` has a very wide range (0–25,691) compared to the PCA features which are already
centred near zero. `Time_sin` and `Time_cos` are already bounded in [−1, 1] — no scaling needed.

**The scaler is fit on `X_train` only**, then applied to both splits.
Fitting on the full dataset would leak test-set statistics (mean, std) into the scaler — a form of data leakage.

In [28]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = X_train.copy()
X_test = X_test.copy()
X_train[['Amount']] = scaler.fit_transform(X_train[['Amount']])
X_test[['Amount']]  = scaler.transform(X_test[['Amount']])

print('Amount in X_train — mean:', X_train['Amount'].mean().round(6),
      ' std:', X_train['Amount'].std().round(6))
print('Amount in X_test  — mean:', X_test['Amount'].mean().round(4),
      ' std:', X_test['Amount'].std().round(4))

Amount in X_train — mean: 0.0  std: 1.000002
Amount in X_test  — mean: 0.0017  std: 1.091


In [29]:
print('X_train:', X_train.shape, '| X_test:', X_test.shape)
print('Features:', X_train.columns.tolist())

X_train: (226980, 20) | X_test: (56746, 20)
Features: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V14', 'V16', 'V17', 'V18', 'V19', 'Amount', 'Time_sin', 'Time_cos']


## 5. Save Processed Splits

Persist the four arrays so `03_modeling.ipynb` can load them directly
without re-running the preprocessing pipeline.

In [30]:
os.makedirs('../data/processed', exist_ok=True)

X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv',   index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv',   index=False)

print('Saved to data/processed/')
print(f'  X_train: {X_train.shape}')
print(f'  X_test:  {X_test.shape}')

Saved to data/processed/
  X_train: (226980, 20)
  X_test:  (56746, 20)


## 5. Handoff to Notebook 3 — What Modeling Must Address

The preprocessed data is now ready. The following decisions and validations are deferred to `03_modeling.ipynb`:

| # | Item | Why deferred |
|---|---|---|
| 1 | **Apply SMOTE on `X_train` only** | SMOTE is a training strategy, not a preprocessing step; synthetic samples must never appear in test set |
| 2 | **Compare SMOTE vs cost-sensitive weighting** | XGBoost supports `scale_pos_weight` as an alternative to resampling — both strategies should be trained and compared directly |
| 3 | **Evaluation metrics** | Use Precision, Recall, F1, AUC-PR — not accuracy  |
| 4 | **Model choice** | Logistic Regression as baseline; Random Forest and XGBoost as tree-based models — each choice should be justified, not just listed |
| 5 | **Threshold analysis** | Default 0.5 threshold is not appropriate for fraud detection; optimal threshold is a business decision based on the cost asymmetry between false negatives and false positives |
| 6 | **Feature importance** | Top features from the best model should be checked against EDA correlation findings (V14, V17, V12, V4, V11) to validate the full analysis narrative |
| 7 | **Cross-validation** *(optional)* | If stability of results is in question, StratifiedKFold can be applied on the training set — but given the dataset size (~228k training rows), a single stratified split is likely sufficient |
| 8 | **`X_test` is locked** | The test set has been split and scaled but never seen by any model. Use it once at the very end — not for tuning |